<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<br>
<b> [DP1] DCR Investigation: Hexbin Plot Generation</b><br>

Contact author: Audrey Budlong <br>
Last verified to run: 12 May 2025 <br>

Goes through a set of LSST ComCam visits from 26 November 2024 to generate a hexbin plot (g-i color vs. angular separation) as a demonstration of the differential chromatic refraction effect.

### Table of Contents

1. Main Package Imports
2. Configure Data Access
3. Determine Astrometry + Photometry Matches
4. Calculate Blackbody Temperatures
5. Calculate the Effective Wavelengths
6. Calculate the Differential Refraction
7. Save Results to CSV
8. Plot Results (Hexbin)

### 1. Main Package Imports

In [ ]:
import lsst.daf.butler as dafButler
import lsst.geom
import math
import matplotlib.patheffects as pathEffects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import warnings
import esutil

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.modeling import models
from astropy.modeling.models import BlackBody
from astropy.table import Table, join, vstack, hstack
from lsst.afw.coord import refraction, differentialRefraction
from lsst.ip.diffim import calculateImageParallacticAngle
from lsst.utils.plotting import set_rubin_plotstyle, stars_cmap, stars_color, accent_color
from scipy import constants
from scipy.optimize import curve_fit

set_rubin_plotstyle()
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
plt.style.use('ggplot')
plt.rcParams['axes.facecolor'] = 'whitesmoke'
plt.rcParams['font.family'] = 'serif'

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

### 2. Configure Data Access

In [ ]:
visits_original = [2024112600111, 2024112600112, 2024112600113, 2024112600114, 2024112600116,
                  2024112600117, 2024112600118, 2024112600119, 2024112600120, 2024112600121,
                  2024112600122, 2024112600149, 2024112600150, 2024112600151, 2024112600152,
                  2024112600153, 2024112600154, 2024112600155, 2024112600156, 2024112600157, 
                  2024112600158, 2024112600169, 2024112600170, 2024112600171, 2024112600172,
                  2024112600173, 2024112600174, 2024112600175, 2024112600176, 2024112600177,
                  2024112600178, 2024112600309, 2024112600310, 2024112600311, 2024112600312,
                  2024112600313, 2024112600314, 2024112600315, 2024112600316, 2024112600317,
                  2024112600318]

In [ ]:
repo = "/repo/main"
collection = "LSSTComCam/runs/DRP/DP1/w_2025_17/DM-50530"
butler_registry = dafButler.Butler(repo)
butler_registry.registry.queryCollections(collection)

butler = dafButler.Butler(repo, collections=butler_registry.registry.queryCollections(collection))
datarefs = butler.query_datasets("preliminary_visit_image")

In [ ]:
refs = []
visits = []
for ref in datarefs:
    if ref.dataId["visit"] in visits_original:
        refs.append(ref)
        visits.append(ref.dataId["visit"])

visits = set(visits)

In [ ]:
visits == visits_original

In [ ]:
wcsList = [butler.get("preliminary_visit_image.wcs", dataId=ref.dataId) for ref in refs]

In [ ]:
visitInfoList = [butler.get("preliminary_visit_image.visitInfo", dataId=ref.dataId) for ref in refs]

### 3. Determine Astrometry + Photometry Matches

In [ ]:
def initialPsfStarsFootprintsDetCat(dataref):
    psfSourceTable = butler.get("single_visit_psf_star_footprints", dataId=dataref.dataId)
    psfsources = psfSourceTable.asAstropy()
    return psfsources

def initialStarsFootprintsDetCat(dataref):
    sourceTable = butler.get("single_visit_star_footprints", dataId=dataref.dataId)
    sources = sourceTable.asAstropy()
    return sources

def astrometryMatchDetCat(dataref):
    # retrieve astrometry match catalog
    astrometryTable = butler.get("initial_astrometry_match_detector", dataId=dataref.dataId)
    astrometry = astrometryTable.asAstropy()

    astrometryLen = len(astrometry)
    visitInfo = butler.get("preliminary_visit_image.visitInfo", dataId=dataref.dataId)
    wcs = butler.get("preliminary_visit_image.wcs", dataId=dataref.dataId)    
    
    # calculating airmass
    airmass = visitInfo.boresightAirmass

    # calculating elevation
    elevation = visitInfo.getBoresightAzAlt().getLatitude()

    # parallactic angle
    boresightParAngle = visitInfo.boresightParAngle

    # observatory info
    observatory = visitInfo.getObservatory()
    
    # calculate difference in reference and source coordinates
    angularSepArcsec = []
    for i in range(astrometryLen):
        c1 = SkyCoord(ra=astrometry['ref_coord_ra'][i]*u.radian, dec=astrometry['ref_coord_dec'][i]*u.radian, frame='icrs')
        c2 = SkyCoord(ra=astrometry['src_coord_ra'][i]*u.radian, dec=astrometry['src_coord_dec'][i]*u.radian, frame='icrs')
        angularSeparation = (c1.separation(c2).radian*u.radian).to(u.arcsec)
        angularSepArcsec.append(angularSeparation)

    # append angular separation between ref and src coords to astrometry catalog
    astrometry['angular_separation'] = angularSepArcsec

    # new method for difference in location
    refX, refY = wcs.skyToPixelArray(ra=astrometry['ref_coord_ra'], dec=astrometry['ref_coord_dec'])
    srcX, srcY = wcs.skyToPixelArray(ra=astrometry['src_coord_ra'], dec=astrometry['src_coord_dec'])

    parallacticAngle = calculateImageParallacticAngle(visitInfo, wcs)

    dx = refX-srcX
    dy = refY-srcY
    amplitude = [np.sqrt(x**2 + y**2) for (x, y) in zip(dx, dy)]
    angleA = [lsst.geom.Angle(np.arctan2(y, x)+(np.pi/2)) for (x, y) in zip(dx, dy)]
    perpendicular = [amp*np.sin(float(anglA-parallacticAngle)) for (amp, anglA) in zip(amplitude, angleA)]
    parallel = [amp*np.cos(float((anglA-parallacticAngle))) for (amp, anglA) in zip(amplitude, angleA)]

    # converting perpendicular and parallel components into wcs
    arcsecPerpendicular = [(lsst.geom.Angle(per)*wcs.getPixelScale()).asArcseconds() for per in perpendicular]
    arcsecParallel = [(lsst.geom.Angle(par)*wcs.getPixelScale()).asArcseconds() for par in parallel]
    
    astrometry['airmass'] = astrometryLen*[airmass,]
    astrometry['elevation'] = astrometryLen*[elevation,]
    astrometry['boresightParAngle'] = astrometryLen*[boresightParAngle,]
    astrometry['observatory'] = astrometryLen*[observatory,]
    astrometry['perpendicular'] = arcsecPerpendicular
    astrometry['parallel'] = arcsecParallel
    
    return astrometry

def photometryMatchDetCat(dataref, band1, band2):
    # retrieve astrometry match catalog
    photometryTable = butler.get("initial_photometry_match_detector", dataId=dataref.dataId)
    photometry = photometryTable.asAstropy()
    photometry = photometry[photometry['src_sky_source']==False]

    # convert nJy to AB mag
    b1_mag = photometry[f'ref_monster_ComCam_{band1}_flux'].to(u.ABmag)
    b2_mag = photometry[f'ref_monster_ComCam_{band2}_flux'].to(u.ABmag)

    # append AB magnitudes
    photometry[f'{band1}Mag'] = b1_mag
    photometry[f'{band2}Mag'] = b2_mag

    # calculate the difference in magnitude and append
    photometry[f'{band2}-{band1} mag'] = b2_mag-b1_mag
    
    return photometry

def preloadedData(repo, collection, dataset, visitN, band):
    butler = dafButler.Butler(repo)
    registry = butler.registry
    registry.getCollectionSummary(collection)
    visit = visitN
    datasetRefs = butler.query_datasets(dataset,
                                        collections=['refcats'],
                                        instrument='LSSTComCam',
                                        visit=visit,
                                        with_dimension_records=True
                                        )
    refCats = [butler.getDeferred(ref) for ref in datasetRefs]
    (record, ) = butler.query_dimension_records('visit', instrument='LSSTComCam', visit=visit)
    center = lsst.geom.SpherePoint(record.region.getCentroid())
    config = LoadReferenceObjectsConfig()
    config.filterMap = {'LSSTComCam': f'{band}'}
        
    refObjLoader = ReferenceObjectLoader(dataIds = [ref.dataId for ref in datasetRefs], refCats=refCats, name=dataset, config=config)
    refCatalog = refObjLoader.loadSkyCircle(center, 1.0*lsst.geom.degrees, f'{band}')
    return refCatalog

def diaSourceTable(dataref):
    diaSources = butler.get('goodSeeingDiff_diaSrcTable', dataId=dataref.dataId)
    return diaSources

def matchedSources(dataref, band1, band2):
    # loading catalogs
    stars = initialStarsFootprintsDetCat(dataref)
    psf_stars = initialPsfStarsFootprintsDetCat(dataref)
    astrometry_matches = astrometryMatchDetCat(dataref)
    photometry_matches = photometryMatchDetCat(dataref, band1, band2)
    
    matched_ids = stars["psf_id"] > 0
    matches = np.searchsorted(psf_stars["id"], stars["psf_id"][matched_ids])
    # psf_stars[matches] and stars[matched_ids] are the matching objects
    
    # only non-zero psf_ids have cross-matches
    # matched_ids = photometry_matches["src_psf_id"] > 0
    matches = esutil.numpy_util.match(astrometry_matches["src_id"], photometry_matches["src_psf_id"])

    astrometry = astrometry_matches[matches[0]]
    photometry = photometry_matches[matches[1]]

    # sort
    astrometry.sort('src_id')
    photometry.sort('src_psf_id')

    # concatenate
    merged = hstack([astrometry, photometry])

    return merged

In [ ]:
resultTables = []
for ref in refs:
    singleVisitResult = matchedSources(ref, 'i', 'g')
    resultTables.append(singleVisitResult)
    print(len(refs)-len(resultTables))

testAllData = vstack(resultTables)
allData = testAllData.to_pandas()

### 4. Calculate Blackbody Temperatures

In [ ]:
def findBlackbodyTemp(dataframe):
    # define the blackbody function
    def bbFit(wavelength, temp, scale):
        tempK = temp * u.K
        blackbody = models.BlackBody(temperature=tempK)
        flux = scale * blackbody(wavelength * u.nm).to(u.nJy / u.sr, equivalencies=u.spectral()).value
        return flux

    # define vaiable for calculated bbTemps
    bbTemps = []

    # iterate through each of the objects in the catalog individually 
    for i in range(len(dataframe)):
        
        # provided data points (update your points if needed)
        gpoint = (478.5, dataframe['ref_monster_ComCam_g_flux_1'][i])
        rpoint = (650.0, dataframe['ref_monster_ComCam_r_flux_1'][i])
        ipoint = (754.6, dataframe['ref_monster_ComCam_i_flux_1'][i])
        zpoint = (910.0, dataframe['ref_monster_ComCam_z_flux_1'][i])

        data = (gpoint, rpoint, ipoint, zpoint)
        
        cleaned_data = [(x, y) for x, y in data if not (math.isnan(y))]

        xdata = [x for x, y in cleaned_data]
        ydata = [y for x, y in cleaned_data]

        bounds = ([1000, 0], [10000, 1e-18])
        initial_guess = [4000, 5*10e-22]

        # Perform the fit with bounds
        popt, pcov = curve_fit(bbFit, xdata, ydata, p0=initial_guess, bounds=bounds) #, method='trf')

        print(len(dataframe) - len(bbTemps))

        bbTemps.append(popt[0])
        
    return bbTemps


In [ ]:
blackbodyTemperatures = findBlackbodyTemp(allData)
allData['bbTemp'] = blackbodyTemperatures

### 5. Calculate the Effective Wavelengths

In [ ]:
def effectiveWavelength(bbtemperatures):

    # defining variable for effective wavelengths
    effWavelengths = []
    
    # iterating through the temperatures of all the objects individually
    for temp in bbtemperatures:
        # define the black body spectrum from temperature
        spectrum = []
        for wavelength in range(400, 551):
            bbModel = models.BlackBody(temperature=temp*u.K)
            bbSpectrum = bbModel(wavelength * u.nm).to(u.nJy / u.sr, equivalencies=u.spectral()).value
            spectrum.append(bbSpectrum)
            
        # finding spectrum sum
        spectrumSum = 0
        spectrumTotal = 0
        for wavelengthIndex, wavelength in enumerate(np.linspace(400, 550, 151)):
            spectrumSum += wavelength*spectrum[wavelengthIndex]
            spectrumTotal += spectrum[wavelengthIndex]
  
        # determine the effective wavelength
        effectiveWavelength = spectrumSum/spectrumTotal

        # append results
        effWavelengths.append(effectiveWavelength)

        print(len(bbtemperatures)-len(effWavelengths))

    # return the effective wavelength
    return effWavelengths

In [ ]:
effWvl = effectiveWavelength(blackbodyTemperatures)

### 6. Calculate the Differential Refraction

In [ ]:
def diffRefraction(dataset, wavelength, refWavelength):
    elevation = dataset['elevation'][0]
    observatory = dataset['observatory'][0]
    diffRef = []
    for w in wavelength:
        refraction = differentialRefraction(w, refWavelength, dataset['elevation'][0], dataset['observatory'][0])
        refractionInArcsec = refraction.asArcseconds()
        diffRef.append(refractionInArcsec)
        print(len(dataset)-len(diffRef))
    return np.array(diffRef)

In [ ]:
refTemp = np.mean(blackbodyTemperatures)
refWavelength = effectiveWavelength([refTemp])

floatEW = [float(i) for i in effWvl]
allData["effective wavelength"] = floatEW

refEW = [float(i) for i in refWavelength]
allData["reference effective wavelength"] = len(allData)*[refEW,]

differentialRefraction_blackbody = diffRefraction(allData, floatEW, refEW[0])
allData["differential refraction"] = differentialRefraction

### 7. Save Results to CSV

In [ ]:
allData.to_csv("allData.csv")

### 8. Plot Results (Hexbin)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2, sharey=True)
plt.subplots_adjust(hspace=0, wspace=0, left=0.12, bottom=0.15)

xlim = differentialRefraction_blackbody.min(), differentialRefraction_blackbody.max()
ylim = allData['g-i mag'].min(), allData['g-i mag'].max()

hb = ax[0,0].hexbin(allData['parallel'], allData['g-i mag'], gridsize=50, cmap=stars_cmap(), mincnt=1)
ax[0,0].set(xlim=(-0.07, 0.07), ylim=ylim)
ax[0,0].set_title("Parallel", fontsize=15)
ax[0,0].axvline(x=0, color=accent_color(), linestyle='--')
ax[0,0].tick_params("x", labelbottom=False)

ax[0,0].text(0.01, 0.4, r'Mag$_{{AB}}$ (g-i)', rotation="vertical", transform=fig.transFigure)
ax[0,0].text(0.35, 0.05, r'Angular Offset (arcsec)', transform=fig.transFigure)

hb = ax[0,1].hexbin(allData['perpendicular'], allData['g-i mag'], gridsize=50, cmap=stars_cmap(), mincnt=1)
ax[0,1].set(xlim=(-0.07, 0.07), ylim=ylim)
ax[0,1].set_title("Perpendicular", fontsize=15)
ax[0,1].axvline(x=0, color=accent_color(), linestyle='--')
ax[0,1].tick_params("x", labelbottom=False)

label = "Number of Sources"                                                                            
axBbox = ax[0,1].get_position()                                                                                    
cax = fig.add_axes([axBbox.x1, axBbox.y0, 0.04, axBbox.y1 - axBbox.y0])                                       
fig.colorbar(hb, cax=cax)                                                                                
text = cax.text(0.5, 0.5, label, color="k", rotation="vertical", transform=cax.transAxes, ha="center",        
                       va="center", fontsize=10)                                                                     
text.set_path_effects([pathEffects.Stroke(linewidth=3, foreground="w"), pathEffects.Normal()])   
# to-do
labels = [item.get_text() for item in cax.get_yticklabels()]
# print(labels)


hb = ax[1,0].hexbin(allData['parallel'], allData['g-i mag'], gridsize=50, bins='log', cmap=stars_cmap(), mincnt=1)
ax[1,0].set(xlim=(-0.07, 0.07), ylim=ylim)
ax[1,0].axvline(x=0, color=accent_color(), linestyle='--')        


hb = ax[1,1].hexbin(allData['perpendicular'], allData['g-i mag'], gridsize=50, bins='log', cmap=stars_cmap(), mincnt=1)
ax[1,1].set(xlim=(-0.07, 0.07), ylim=ylim)
ax[1,1].axvline(x=0, color=accent_color(), linestyle='--')
label2 = "Log(Number of Sources)"                                                                            
axBbox = ax[1,1].get_position()                                                                                    
cax = fig.add_axes([axBbox.x1, axBbox.y0, 0.04, axBbox.y1 - axBbox.y0])                                       
fig.colorbar(hb, cax=cax)                                                                                
text = cax.text(0.5, 0.5, label2, color="k", rotation="vertical", transform=cax.transAxes, ha="center",        
                       va="center", fontsize=10)                                                                     
text.set_path_effects([pathEffects.Stroke(linewidth=3, foreground="w"), pathEffects.Normal()])   

plt.savefig('dcr_starsCmap.pdf')
plt.show()